# Supplementary Figure 16 | Annotation enrichments for lncRNA GWAS-QTL colocalization.

**a.** Logistic odds ratios (ORs) for membership of annotated lncRNAs in a GWAS-QTL CoS (GWAS p < 10-5), comparing odds with and without adjustment for ascertainment confounders as covariates: numbers of tissues tested for eQTL and sQTL, and expression as log10(TPM + 1) where TPM is the maximum median tissue TPM. Categories are ordered as in Fig. 4e. Points, ORs; error bars, 95% CIs; asterisks, two-sided Wald p (* p < 0.05; ** p < 0.01; *** p < 0.001).

**b.** Same as (a) for ascertainment-adjusted models stratified by existing versus novel lncRNAs.


## Setup

In [ ]:
import gzip
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from matplotlib.lines import Line2D
from tqdm import tqdm

plt.rc("font", size=6)
plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# Histology-subtype tissues excluded from the median-TPM table (as in Figure 4; cell lines kept)
histology_subtypes = {
    "Colon - Transverse - Muscularis",
    "Stomach - Mixed Cell",
    "Pancreas - Mixed Cell",
    "Small Intestine - Terminal Ileum - Mixed Cell",
    "Stomach - Mucosa",
    "Stomach - Muscularis",
    "Pancreas - Acini",
    "Small Intestine - Terminal Ileum - Lymphoid Aggregate",
    "Colon - Transverse - Mixed Cell",
    "Colon - Transverse - Mucosa",
    "Liver - Mixed Cell",
    "Pancreas - Islets",
    "Liver - Hepatocyte",
    "Liver - Portal Tract",
}
expected_n_bulk_tissues = 54

gwas_p_cut_1e5 = 1e-5
gwas_neglog10p_1e5 = -np.log10(gwas_p_cut_1e5)
annot_group = "annotation (unadjusted)"
joint_group = "annotation + expr. + N tissues"  # one-at-a-time + ascertainment covars
joint_covars = "log_max_tpm + n_tissues_tested_eqtl + n_tissues_tested_sqtl"
fig_width, fig_height, fig_dpi = 7.2, 5.0, 300

qtl_label_for_figure = {
    "no_qtl": "no_qtl",
    "non-lncRNA": "non-lncRNA",
    "Shared": "Shared",
    "lncRNA-specific (weak)": "Shared",
    "lncRNA-specific (robust)": "lncRNA-specific",
}

ensg_stem = re.compile(r"(ENSG\d+)")
paired = sns.color_palette("Paired", 12)
biotype_palette = {"existing": paired[4], "novel": paired[5]}

# Zenodo ColocBoost helpers
def tissue_from_cos_id(cid):
    s = str(cid)
    i = s.rfind("_")
    return s[:i] if i > 0 else s


def lead_from_lists(variants, vcp):
    vs = list(variants)
    vcps = list(vcp)
    ps = pd.to_numeric(pd.Series(vcps), errors="coerce")
    if len(ps) == len(vs) and ps.notna().any():
        return vs[int(ps.values.argmax())]
    return vs[0]


def cos_to_long(row):
    vs = list(row["variants"])
    ps = list(row["vcp"])
    return pd.DataFrame({
        "variant_id": vs,
        "vcp": ps[: len(vs)] + [float("nan")] * (len(vs) - len(ps)),
    })


def _coloc_list_len(val):
    if val is None:
        return 0
    if isinstance(val, float) and np.isnan(val):
        return 0
    if isinstance(val, np.ndarray):
        return len(val)
    if isinstance(val, (list, tuple, set)):
        return len(val)
    return 0


def _typeset(vals):
    if vals is None:
        return set()
    if isinstance(vals, float) and np.isnan(vals):
        return set()
    if isinstance(vals, np.ndarray):
        items = vals.tolist()
    elif isinstance(vals, (list, tuple, set)):
        items = list(vals)
    else:
        items = [vals]
    out = set()
    for v in items:
        if v is None:
            continue
        s = str(v).strip()
        if s and s not in ("nan", "None", "<NA>"):
            out.add(s)
    return out


def _member_qtl_class(types):
    t = _typeset(types)
    lnc = t & {"novel_lncRNA", "existing_lncRNA"}
    non = t - {"novel_lncRNA", "existing_lncRNA"}
    if not lnc:
        return "non-lncRNA"
    if non:
        return "Shared"
    return "lncRNA-specific (robust)"


def _locus_qtl_label(member_classes):
    labs = set(member_classes)
    if labs == {"lncRNA-specific (robust)"}:
        return "lncRNA-specific (robust)"
    if "Shared" in labs or ("non-lncRNA" in labs and "lncRNA-specific (robust)" in labs):
        return "Shared"
    return "non-lncRNA"


def load_main_colocboost(colocboost_dir, *, cols=None):
    frames = []
    for path in sorted(colocboost_dir.glob("*.ColocBoost.parquet")):
        if path.name.startswith("cross_"):
            continue
        tissue = path.name.replace(".ColocBoost.parquet", "")
        df = pd.read_parquet(path, columns=cols)
        df["tissue"] = tissue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def build_cross_tissue_members(cross_tissue_loci_path, colocboost_dir, *, include_ucos=False):
    loci = pd.read_parquet(cross_tissue_loci_path)
    all_cos = load_main_colocboost(colocboost_dir).set_index("cos_id", drop=False)
    memb = loci.explode("member_cos_ids").rename(columns={"member_cos_ids": "cos_id"})
    memb["cos_id"] = memb["cos_id"].astype(str)
    cos_rows = memb[["locus_id", "cos_id"]].merge(
        all_cos.reset_index(drop=True), on="cos_id", how="inner"
    )
    cos_rows["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(cos_rows["variants"], cos_rows["vcp"])
    ]
    if "phenotype_id" not in cos_rows.columns:
        cos_rows["phenotype_id"] = cos_rows["phenotype_ids"]

    qtl_by_locus = (
        cos_rows.groupby("locus_id")["gene_types_robust"]
        .agg(lambda s: _locus_qtl_label(_member_qtl_class(x) for x in s))
    )
    locus_summary = loci.copy()
    locus_summary["qtl_label"] = locus_summary["locus_id"].map(qtl_by_locus)

    if not include_ucos:
        return locus_summary, cos_rows

    member_ids = set(memb["cos_id"])
    has_gwas = all_cos["gwas_traits"].map(_coloc_list_len) > 0
    has_qtl = all_cos["gene_ids_robust"].map(_coloc_list_len) > 0
    ucos = all_cos.loc[
        ~all_cos.index.isin(member_ids) & (has_gwas ^ has_qtl)
    ].reset_index(drop=True)
    ucos = ucos.loc[has_gwas.reindex(ucos["cos_id"]).fillna(False).values].copy()
    ucos["locus_id"] = "ucos_" + ucos["cos_id"].astype(str)
    ucos["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(ucos["variants"], ucos["vcp"])
    ]
    if "phenotype_id" not in ucos.columns:
        ucos["phenotype_id"] = ucos["phenotype_ids"]
    members = pd.concat([cos_rows, ucos], ignore_index=True)
    return locus_summary, members



## Paths

In [ ]:
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
lnc_cat_file = Path("path/to/lncrna_category.txt")
tissue_names_file = Path("annotation/refs/tissue_name.txt")
median_tpm_file = Path("path/to/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz")
eqtl_dir = Path("path/to/v11_eqtl_susie")
sqtl_dir = Path("path/to/v11_sqtl_susie")

colocboost_dir = Path("path/to")
loci_path = Path("path/to/cross_tissue_loci.parquet")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_16_coloc_annotation_ors.pdf"
fig_png = fig_dir / "supp_figure_16_coloc_annotation_ors.png"


## Load data

### Gene classification

In [ ]:
gencode_genes = pd.read_table(gencode_class, low_memory=False)
new_lncrna_ids = set(
    gencode_genes.loc[gencode_genes["gene_type_grouped"] == "novel_lncRNA", "gene_id_stable"]
)
existing_lncrna_ids = set(
    gencode_genes.loc[gencode_genes["gene_type_grouped"] == "existing_lncRNA", "gene_id_stable"]
)
lncrna_ids = new_lncrna_ids | existing_lncrna_ids
stable_to_v47 = (
    gencode_genes.drop_duplicates("gene_id_stable").set_index("gene_id_stable")["gene_id_v47"]
)
print(f"Novel lncRNAs: {len(new_lncrna_ids):,}; existing: {len(existing_lncrna_ids):,}")



### GTEx v11 gene median TPM

Official portal `*_gene_median_tpm.gct.gz`; max across tissues used for ascertainment-adjusted ORs.

In [ ]:
def load_gtex_portal_median_tpm(path: Path) -> pd.DataFrame:
    """Load portal gene_median_tpm GCT; columns = pretty SMTSD names."""
    with gzip.open(path, "rt") as fh:
        fh.readline()
        fh.readline()
        raw = pd.read_csv(fh, sep="\t")
    med = raw.set_index("Name").drop(columns=["Description"], errors="ignore")
    tissue_map = pd.read_csv(tissue_names_file, sep="\t")
    stem_to_pretty = dict(zip(tissue_map["tissue_stem"], tissue_map["tissue_name"]))
    renamed = {}
    for c in med.columns:
        pretty = stem_to_pretty.get(c) or stem_to_pretty.get(c.replace("-", "_"))
        if pretty is None:
            raise KeyError(f"Cannot map portal median tissue column {c!r} to SMTSD")
        renamed[c] = pretty
    return med.rename(columns=renamed)


median_tpm = load_gtex_portal_median_tpm(median_tpm_file)
_before = median_tpm.shape[1]
median_tpm = median_tpm[[c for c in median_tpm.columns if c not in histology_subtypes]]
if median_tpm.shape[1] != expected_n_bulk_tissues:
    raise ValueError(
        f"Expected {expected_n_bulk_tissues} bulk tissues after dropping histology subtypes, "
        f"found {median_tpm.shape[1]} (from {_before} portal columns)"
    )
max_tpm_by_v47 = median_tpm.max(axis=1)
print(
    f"Official median TPM: {median_tpm.shape[0]:,} genes × {median_tpm.shape[1]} bulk tissues "
    f"(dropped {_before - median_tpm.shape[1]} histology subtypes from {_before} portal columns)"
)



### GTEx v11 eGenes (tissues tested)

Per-tissue `*.v11.eGenes.txt.gz` from the portal eQTL release.

In [ ]:
# Ascertainment covariate: count tissues where each gene appears in eGenes.
egene_files = sorted(eqtl_dir.glob("*.v11.eGenes.txt.gz"))
if not egene_files:
    raise FileNotFoundError(f"No *.v11.eGenes.txt.gz under {eqtl_dir}")
_egene_rows = []
for fpath in tqdm(egene_files, desc="eGenes tissues"):
    tissue = fpath.name.split(".v11.")[0]
    genes = pd.read_csv(fpath, sep="\t", usecols=["gene_id"])
    genes["gene_id_stable"] = genes["gene_id"].astype(str).str.split(".").str[0]
    genes["tissue"] = tissue
    _egene_rows.append(genes[["gene_id_stable", "tissue"]])
_egene_gt = pd.concat(_egene_rows, ignore_index=True)
n_eqtl_tested_by_stable = _egene_gt.groupby("gene_id_stable")["tissue"].nunique()
del _egene_rows, _egene_gt
print(f"eGenes files: {len(egene_files)}; genes with tissues-tested: {len(n_eqtl_tested_by_stable):,}")



### GTEx v11 sGenes (tissues tested)

Per-tissue `*.v11.sGenes.txt.gz` from the portal sQTL release.

In [ ]:
# Ascertainment covariate: count tissues where each gene appears in sGenes.
sgene_files = sorted(sqtl_dir.glob("*.v11.sGenes.txt.gz"))
if not sgene_files:
    raise FileNotFoundError(f"No *.v11.sGenes.txt.gz under {sqtl_dir}")
_sgene_rows = []
for fpath in tqdm(sgene_files, desc="sGenes tissues"):
    tissue = fpath.name.split(".v11.")[0]
    genes = pd.read_csv(fpath, sep="\t", usecols=["gene_id"])
    genes["gene_id_stable"] = genes["gene_id"].astype(str).str.split(".").str[0]
    genes["tissue"] = tissue
    _sgene_rows.append(genes[["gene_id_stable", "tissue"]].drop_duplicates())
_sgene_gt = pd.concat(_sgene_rows, ignore_index=True)
n_sqtl_tested_by_stable = _sgene_gt.groupby("gene_id_stable")["tissue"].nunique()
del _sgene_rows, _sgene_gt
print(f"sGenes files: {len(sgene_files)}; genes with tissues-tested: {len(n_sqtl_tested_by_stable):,}")



### Coloc locus tables

In [ ]:
locus_summary, locus_long = build_cross_tissue_members(loci_path, colocboost_dir, include_ucos=False)
locus_summary["qtl_label_raw"] = locus_summary["qtl_label"].astype(str)
locus_summary["qtl_label"] = (
    locus_summary["qtl_label_raw"].map(qtl_label_for_figure).fillna(locus_summary["qtl_label_raw"])
)
print(
    f"Loci: {len(locus_summary):,}; members: {len(locus_long):,}; "
    f"GWS: {int((locus_summary['gwas_gws_traits'].map(len) > 0).sum()):,}"
)
print(locus_summary["qtl_label"].value_counts().to_string())


### Annotation inputs

In [ ]:
lnc_cat_all = pd.read_table(
    lnc_cat_file,
    usecols=[
        "gene_id_stable", "is_mirna_host", "is_ncORF",
        "expressed_dgtex", "expressed_macaque", "expressed_marmoset",
        "is_tissue_specific", "peak_tissue",
        "macaque_tissue_concordant", "marmoset_tissue_concordant",
        "primate_cons_exon_unique", "primate_cons_sj_unique",
        "tss_e2g_enh",
    ],
).drop_duplicates("gene_id_stable")
lnc_cat_all = lnc_cat_all[lnc_cat_all["gene_id_stable"].isin(lncrna_ids)].copy()
lnc_by_stable = lnc_cat_all.set_index("gene_id_stable")
print(
    f"lncRNA categories: {len(lnc_cat_all):,}; "
    f"tss_e2g_enh={int(lnc_by_stable['tss_e2g_enh'].sum()):,}; "
    f"primate_cons_exon_unique={int(lnc_by_stable['primate_cons_exon_unique'].sum()):,}"
)



## Run analysis

### Annotation traits

In [ ]:
nontestis_ok = ~lnc_by_stable["peak_tissue"].eq("Testis")
# expressed_macaque/marmoset already require an ortholog id (classify_lncrna)
primate_expr_nontestis = (
    lnc_by_stable["expressed_macaque"].astype(bool)
    | lnc_by_stable["expressed_marmoset"].astype(bool)
) & nontestis_ok
dgtex = lnc_by_stable["expressed_dgtex"].astype(bool)
mac_tc = lnc_by_stable["macaque_tissue_concordant"].astype(bool)
mar_tc = lnc_by_stable["marmoset_tissue_concordant"].astype(bool)
ts = lnc_by_stable["is_tissue_specific"].astype(bool)
testis_peak = lnc_by_stable["peak_tissue"].eq("Testis")
ts_testis = ts & testis_peak
ts_nontestis = ts & nontestis_ok
nhp_match = mac_tc | mar_tc
tc_testis = ts_testis & nhp_match
tc_nontestis = ts_nontestis & nhp_match

primate_cons_exon_unique = lnc_by_stable["primate_cons_exon_unique"].astype(bool)
primate_cons_sj_unique = lnc_by_stable["primate_cons_sj_unique"].astype(bool)
tss_e2g_enh = lnc_by_stable["tss_e2g_enh"].astype(bool)

or_traits = [
    ("tss_e2g_enh", "enhancer", None),
    ("is_mirna_host", "miRNA host", None),
    ("is_ncORF", "translated ncORF", None),
    ("primate_cons_exon_unique", "primate\nexon cons.", None),
    ("primate_cons_sj_unique", "primate\nsplice junct. cons.", None),
    ("primate_expr", "primate expr.", None),
    ("dev_expr", "dev. expr.", None),
    ("ts_testis", "TS (testis)", "ts_vs_not_ts_testis"),
    ("ts_nontestis_only", "TS (non-testis)", "ts_vs_not_ts_nontestis"),
    ("tc_testis", "TC (testis)", "ts_vs_not_tc_testis"),
    ("tc_nontestis", "TC (non-testis)", "ts_vs_not_tc_nontestis"),
]
print(f"{len(or_traits)} annotation traits")



### CoS gene sets at p < 1×10⁻⁵

Existing / novel coloc genes = lncRNAs in ≥1 Shared or lncRNA-specific CoS (after weak→Shared remapping).

In [ ]:
def as_stem_set(vals):
    out = set()
    if vals is None:
        return out
    try:
        if isinstance(vals, float) and np.isnan(vals):
            return out
    except (TypeError, ValueError):
        pass
    items = vals if isinstance(vals, (list, tuple, np.ndarray, set)) else [vals]
    for v in items:
        if v is None:
            continue
        m = ensg_stem.search(str(v))
        if m:
            out.add(m.group(1))
    return out


def has_gwas_at_cut(phenotype_id, max_neg_log10_p, cut=gwas_neglog10p_1e5):
    parts = [p.strip() for p in str(phenotype_id).split(";") if p.strip()]
    vals = []
    if max_neg_log10_p is not None and not (isinstance(max_neg_log10_p, float) and np.isnan(max_neg_log10_p)):
        for tok in str(max_neg_log10_p).split(","):
            tok = tok.strip()
            if not tok or tok.lower() == "nan":
                vals.append(np.nan)
            else:
                try:
                    vals.append(float(tok))
                except ValueError:
                    vals.append(np.nan)
    for i, part in enumerate(parts):
        if i >= len(vals):
            break
        pl = part.lower()
        if "expression" in pl or "splicing" in pl:
            continue
        v = vals[i]
        if np.isfinite(v) and float(v) > cut:
            return True
    return False


member_1e5 = [
    has_gwas_at_cut(pid, csmax)
    for pid, csmax in tqdm(
        zip(locus_long["phenotype_ids"], locus_long["cos_max_pval_nominal_phenotypes"]),
        total=len(locus_long),
        desc="p<1e-5 members",
    )
]
locus_long_1e5 = locus_long[np.asarray(member_1e5, dtype=bool)].copy()
loci_1e5_ids = set(locus_long_1e5["locus_id"].astype(str))
all_loci = locus_summary[locus_summary["locus_id"].astype(str).isin(loci_1e5_ids)].copy()
all_loci = all_loci[all_loci["qtl_label"].astype(str) != "no_qtl"].copy()
all_loci["gene_stems"] = all_loci["gene_ids_robust"].map(as_stem_set)
print(f"p<1e-5 gene-coloc loci: {len(all_loci):,}")


def lnc_genes_in_qtl_label(loci_df, label, biotype_ids):
    genes = set()
    sub = loci_df[loci_df["qtl_label"].astype(str) == label]
    for r in sub.itertuples(index=False):
        genes |= set(r.gene_stems) & biotype_ids
    return genes


existing_coloc = (
    lnc_genes_in_qtl_label(all_loci, "Shared", existing_lncrna_ids)
    | lnc_genes_in_qtl_label(all_loci, "lncRNA-specific", existing_lncrna_ids)
)
novel_coloc = (
    lnc_genes_in_qtl_label(all_loci, "Shared", new_lncrna_ids)
    | lnc_genes_in_qtl_label(all_loci, "lncRNA-specific", new_lncrna_ids)
)
# lncRNA-specific genes, used only for the category order (as in the main-text Figure 4 OR panel)
specific_genes = lnc_genes_in_qtl_label(all_loci, "lncRNA-specific", lncrna_ids)
print(
    f"coloc genes: existing={len(existing_coloc):,}; novel={len(novel_coloc):,}; "
    f"lncRNA-specific={len(specific_genes):,}"
)



### Odds ratios by biotype

In [ ]:
def build_outcome_gld(outcome_ids, outcome_col, universe_ids):
    gld = lnc_cat_all[lnc_cat_all["gene_id_stable"].isin(universe_ids)].copy()
    gld[outcome_col] = gld["gene_id_stable"].isin(outcome_ids).astype(int)
    gene_v47 = gld["gene_id_stable"].map(stable_to_v47)
    gld["log_max_tpm"] = np.log10(gene_v47.map(max_tpm_by_v47).fillna(0) + 1)
    gld["n_tissues_tested_eqtl"] = (
        gld["gene_id_stable"].map(n_eqtl_tested_by_stable).fillna(0).astype(float)
    )
    gld["n_tissues_tested_sqtl"] = (
        gld["gene_id_stable"].map(n_sqtl_tested_by_stable).fillna(0).astype(float)
    )
    gld["tss_e2g_enh"] = gld["gene_id_stable"].map(tss_e2g_enh).fillna(False).astype(int)
    gld["is_mirna_host"] = gld["is_mirna_host"].astype(int)
    gld["is_ncORF"] = gld["is_ncORF"].astype(int)
    gld["primate_cons_exon_unique"] = (
        gld["gene_id_stable"].map(primate_cons_exon_unique).fillna(False).astype(int)
    )
    gld["primate_cons_sj_unique"] = (
        gld["gene_id_stable"].map(primate_cons_sj_unique).fillna(False).astype(int)
    )
    gld["primate_expr"] = gld["gene_id_stable"].map(primate_expr_nontestis).fillna(False).astype(int)
    gld["dev_expr"] = gld["gene_id_stable"].map(dgtex).fillna(False).astype(int)

    _ts_testis = gld["gene_id_stable"].map(ts_testis).fillna(False).astype(int)
    _ts_nontestis = gld["gene_id_stable"].map(ts_nontestis).fillna(False).astype(int)
    gld["tc_testis"] = gld["gene_id_stable"].map(tc_testis).fillna(False).astype(int)
    gld["tc_nontestis"] = gld["gene_id_stable"].map(tc_nontestis).fillna(False).astype(int)
    gld["ts_testis"] = ((_ts_testis == 1) & (gld["tc_testis"] == 0)).astype(int)
    gld["ts_nontestis_only"] = ((_ts_nontestis == 1) & (gld["tc_nontestis"] == 0)).astype(int)
    gld["ts_vs_not_ts_testis"] = (
        (gld["ts_nontestis_only"] == 0) & (gld["tc_testis"] == 0) & (gld["tc_nontestis"] == 0)
    ).astype(int)
    gld["ts_vs_not_ts_nontestis"] = (
        (gld["ts_testis"] == 0) & (gld["tc_testis"] == 0) & (gld["tc_nontestis"] == 0)
    ).astype(int)
    gld["ts_vs_not_tc_testis"] = (
        (gld["ts_testis"] == 0) & (gld["ts_nontestis_only"] == 0) & (gld["tc_nontestis"] == 0)
    ).astype(int)
    gld["ts_vs_not_tc_nontestis"] = (
        (gld["ts_testis"] == 0) & (gld["ts_nontestis_only"] == 0) & (gld["tc_testis"] == 0)
    ).astype(int)
    return gld


def fit_trait_or(work, outcome_col, trait_col, subset_col=None, covars=""):
    d = work[work[subset_col] == 1].copy() if subset_col is not None else work.copy()
    if d.empty or d[outcome_col].nunique() < 2 or d[trait_col].nunique() < 2:
        return None
    n_pos = int(d[outcome_col].sum())
    if n_pos < 3 or len(d) - n_pos < 3:
        return None
    is_binary = set(pd.unique(d[trait_col].dropna())).issubset({0, 1, 0.0, 1.0, True, False})
    if is_binary:
        pos = d[outcome_col] == 1
        n_pos_trait = int(d.loc[pos, trait_col].sum())
        if d[trait_col].sum() < 3 or n_pos_trait == 0 or (n_pos - n_pos_trait) == 0:
            return None
    covar_cols = [c.strip() for c in covars.split("+")] if covars else []
    for c in covar_cols:
        if c not in d.columns or d[c].nunique() < 2:
            return None
    formula = f"{outcome_col} ~ {trait_col}" + (f" + {covars}" if covars else "")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = smf.logit(formula, data=d).fit(disp=False, maxiter=200)
    except Exception:
        return None
    if trait_col not in fit.params.index:
        return None
    ci = fit.conf_int().loc[trait_col]
    or_hat = float(np.exp(fit.params[trait_col]))
    ci_low, ci_high = float(np.exp(ci.iloc[0])), float(np.exp(ci.iloc[1]))
    if not all(np.isfinite(v) for v in (or_hat, ci_low, ci_high)):
        return None
    return {
        "trait": trait_col, "OR": or_hat, "CI_low": ci_low, "CI_high": ci_high,
        "pvalue": float(fit.pvalues[trait_col]), "n_pos": n_pos,
    }


def compute_individual_ors(gld, outcome_col, traits, group, covars=""):
    rows = []
    for trait_col, trait_label, subset_col in traits:
        row = fit_trait_or(
            gld, outcome_col, trait_col, subset_col=subset_col, covars=covars
        ) or {
            "trait": trait_col, "OR": np.nan, "CI_low": np.nan, "CI_high": np.nan,
            "pvalue": np.nan,
            "n_pos": int(gld[outcome_col].sum()) if subset_col is None else int(
                gld.loc[gld[subset_col] == 1, outcome_col].sum()
            ),
        }
        row["label"] = trait_label
        row["group"] = group
        rows.append(row)
    return pd.DataFrame(rows)


biotype_or_rows = []
for bio_name, bio_ids, coloc_ids in [
    ("existing", existing_lncrna_ids, existing_coloc),
    ("novel", new_lncrna_ids, novel_coloc),
]:
    gld = build_outcome_gld(coloc_ids, "has_coloc", bio_ids)
    adj = compute_individual_ors(
        gld, "has_coloc", or_traits, joint_group, covars=joint_covars
    )
    adj["biotype"] = bio_name
    biotype_or_rows.append(adj)
    print(
        f"{bio_name}: n={len(gld):,}, n_coloc={int(gld['has_coloc'].sum()):,}, "
        f"adj covars={joint_covars}"
    )

biotype_or_df = pd.concat(biotype_or_rows, ignore_index=True)

# Category order as in the main-text Figure 4 OR panel: sort by lncRNA-specific CoS-class OR (high → low)
gld_specific = build_outcome_gld(specific_genes, "has_class", lncrna_ids)
spec_or_df = compute_individual_ors(
    gld_specific, "has_class", or_traits, joint_group, covars=joint_covars
)
_spec_or_by_trait = spec_or_df.drop_duplicates("trait").set_index("trait")["OR"]

def _spec_trait_sort_key(trait_col):
    if trait_col not in _spec_or_by_trait.index or not np.isfinite(_spec_or_by_trait[trait_col]):
        return (1, 0.0)
    return (0, -float(_spec_or_by_trait[trait_col]))

_traits_sorted = sorted(or_traits, key=lambda tls: _spec_trait_sort_key(tls[0]))
cat_order = [lab for _t, lab, _s in _traits_sorted]
trait_order = [t for t, _lab, _s in _traits_sorted]
print(
    f"lncRNA-specific (for cat order): n={len(gld_specific):,}, "
    f"n_pos={int(gld_specific['has_class'].sum()):,}"
)
print("Category order (by lncRNA-specific OR):", trait_order)

# Panel a: pooled across all annotated lncRNAs (unadjusted vs ascertainment-adjusted)
all_coloc = existing_coloc | novel_coloc
gld_pooled = build_outcome_gld(all_coloc, "has_coloc", lncrna_ids)
pooled_indiv = compute_individual_ors(
    gld_pooled, "has_coloc", or_traits, annot_group
)
pooled_adj = compute_individual_ors(
    gld_pooled, "has_coloc", or_traits, joint_group, covars=joint_covars
)
pooled_or_df = pd.concat([pooled_indiv, pooled_adj], ignore_index=True)
print(
    f"pooled: n={len(gld_pooled):,}, n_coloc={int(gld_pooled['has_coloc'].sum()):,}, "
    f"adj covars={joint_covars}"
)



## Make figure

In [ ]:
def sig_stars(p):
    if not np.isfinite(p):
        return ""
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def draw_horizontal_forest(ax, plot_df, cat_order, series_specs, xlabel, ymap=None, show_ylabels=True):
    if ymap is None:
        ymap = {lab: i for i, lab in enumerate(cat_order)}
    or_lo_min, or_hi_max = 1.0, 1.0
    legend_handles = []
    for filt, color, marker, legend_lab, dy in series_specs:
        sub = plot_df.copy()
        for k, v in filt.items():
            sub = sub[sub[k] == v]
        sub = sub.drop_duplicates("label").set_index("label")
        xs, ys, xlo, xhi, stars = [], [], [], [], []
        for lab in cat_order:
            y = ymap[lab] + dy
            if lab not in sub.index:
                ax.plot(0.0, y, marker="x", color="0.55", ms=2.5, mew=0.7, zorder=3)
                continue
            row = sub.loc[lab]
            or_hat, lo, hi, p = row["OR"], row["CI_low"], row["CI_high"], row["pvalue"]
            if not (np.isfinite(or_hat) and np.isfinite(lo) and np.isfinite(hi) and or_hat > 0):
                ax.plot(0.0, y, marker="x", color="0.55", ms=2.5, mew=0.7, zorder=3)
                continue
            lx, ll, lh = np.log2(or_hat), np.log2(lo), np.log2(hi)
            xs.append(lx); ys.append(y); xlo.append(ll); xhi.append(lh)
            stars.append(sig_stars(p))
            or_lo_min = min(or_lo_min, lo)
            or_hi_max = max(or_hi_max, hi)
        if xs:
            ax.errorbar(
                xs, ys,
                xerr=[np.array(xs) - np.array(xlo), np.array(xhi) - np.array(xs)],
                fmt=marker, color=color, ecolor=color,
                markerfacecolor=color, markeredgecolor="k", markeredgewidth=0.25,
                elinewidth=0.65, markersize=2.8, zorder=3,
            )
            for x, y, s, hi in zip(xs, ys, stars, xhi):
                if s:
                    ax.text(hi + 0.04, y, s, ha="left", va="center", color="black", fontsize=3.2)
        legend_handles.append(
            Line2D(
                [0], [0], marker=marker, color=color, markerfacecolor=color,
                markeredgecolor="k", markeredgewidth=0.25, linestyle="None",
                markersize=3.2, label=legend_lab,
            )
        )
    ax.axvline(0.0, color="0.5", ls="-", lw=0.45, zorder=1)
    for i in range(len(cat_order)):
        if i % 2 == 0:
            ax.axhspan(i - 0.5, i + 0.5, color="#f7f7f7", zorder=0)
    ax.set_yticks(list(ymap.values()))
    if show_ylabels:
        ax.set_yticklabels(list(ymap.keys()), fontsize=4.2)
    else:
        ax.set_yticklabels([])
    ax.set_ylim(len(cat_order) - 0.5, -0.5)
    ax.set_xlabel(xlabel, fontsize=5.0)
    lo = max(or_lo_min, 1e-6)
    hi = max(or_hi_max, 1.0)
    pad = 1.35
    x0, x1 = np.log2(lo / pad), np.log2(hi * pad)
    k0, k1 = int(np.floor(x0)), int(np.ceil(x1))
    ticks = list(range(k0, k1 + 1))
    ax.set_xticks(ticks)
    ax.set_xticklabels([rf"$2^{{{k}}}$" if k != 0 else "1" for k in ticks], fontsize=4.5)
    ax.set_xlim(x0, x1)
    ax.tick_params(axis="y", length=0, pad=1)
    ax.tick_params(axis="x", labelsize=4.5, pad=1, length=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    return legend_handles


ymap = {lab: i for i, lab in enumerate(cat_order)}
xlabel = "OR lncRNA GWAS-QTL coloc"
pooled_palette = {
    annot_group: "#7A7A7A",
    joint_group: "#2F2F2F",
}

fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
outer = fig.add_gridspec(
    1, 2, width_ratios=[1.15, 1],
    wspace=0.18,
    left=0.22, right=0.99, top=0.86, bottom=0.12,
)

ax_a = fig.add_subplot(outer[0, 0])
handles_a = draw_horizontal_forest(
    ax_a,
    pooled_or_df,
    cat_order,
    [
        ({"group": annot_group}, pooled_palette[annot_group], "o",
         annot_group, -0.12),
        ({"group": joint_group}, pooled_palette[joint_group], "o",
         joint_group, 0.12),
    ],
    xlabel=xlabel,
    ymap=ymap,
    show_ylabels=True,
)
ax_a.legend(
    handles=handles_a, loc="lower center", bbox_to_anchor=(0.5, 1.01),
    ncol=2, frameon=False, fontsize=3.0, handletextpad=0.25, labelspacing=0.2,
    columnspacing=0.8,
)

ax_b = fig.add_subplot(outer[0, 1])
handles_b = draw_horizontal_forest(
    ax_b,
    biotype_or_df,
    cat_order,
    [
        ({"biotype": "existing", "group": joint_group}, biotype_palette["existing"], "o",
         f"existing · {joint_group}", -0.12),
        ({"biotype": "novel", "group": joint_group}, biotype_palette["novel"], "o",
         f"novel · {joint_group}", 0.12),
    ],
    xlabel=xlabel,
    ymap=ymap,
    show_ylabels=False,
)
ax_b.legend(
    handles=handles_b, loc="lower center", bbox_to_anchor=(0.5, 1.01),
    ncol=2, frameon=False, fontsize=3.0, handletextpad=0.25, labelspacing=0.2,
    columnspacing=0.8,
)

_inv = fig.transFigure.inverted()
for ax, label, xoff in [(ax_a, "a", -0.08), (ax_b, "b", -0.02)]:
    _x, _y = _inv.transform(ax.transAxes.transform((xoff, 1.02)))
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)

plt.show()



## Save figure

In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")



## Print stats

In [ ]:
# No numeric values unique to this figure are cited in the paper;
# Fig. 4e owns the ascertainment-adjusted developmental / tissue-concordant ORs.
